# 03 · NDVI from RED + NIR

The core vegetation-index workflow:

```
clip raster to AOI -> extract RED + NIR -> NDVI = (NIR - RED) / (NIR + RED)
                   -> statistics -> vegetation map
```

Band selection is always reported: Vibes Lite never silently assumes a band
order without telling you. NDRE, EVI and SAVI run through the same code path.

In [ ]:
%matplotlib inline
# --- Vibes Lite bootstrap -----------------------------------------------------
# These notebooks drive the processing engines directly (no HTTP layer needed).
import sys
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import config, data, geospatial, preprocessing, indices, analysis
import opencv as vl_cv          # Vibes Lite's own engine (it wraps cv2 internally)
import segmentation, timeseries, visualization, storage

PATHS = config.ensure_directories()
NOTEBOOK_OUT = PATHS["outputs"] / "notebooks"
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)

print("project root :", ROOT)
print("raw data     :", PATHS["raw_data"].relative_to(ROOT))
print("notebook out :", NOTEBOOK_OUT.relative_to(ROOT))

In [ ]:
DEMO_NAME = "synthetic_fields"


def demo_scene(name: str = DEMO_NAME, size: int = 900):
    """Return (raster, aoi, csv) for the labelled synthetic demo scene.

    The scene is only generated when missing. It is explicitly synthetic - its
    GeoTIFF tags carry SYNTHETIC=true - so nothing here pretends to be a real
    satellite acquisition.
    """
    raster = PATHS["raw_data"] / f"{name}.tif"
    aoi = PATHS["raw_data"] / f"{name}.geojson"
    table = PATHS["raw_data"] / f"{name}_ndvi_history.csv"
    if not raster.exists() or not aoi.exists():
        scene = data.generate_synthetic_scene(name=name, size=size, overwrite=True)
        return scene["paths"]["raster"], scene["paths"]["aoi"], scene["paths"]["table"]
    return raster, aoi, table


RASTER, AOI_FILE, HISTORY_CSV = demo_scene()
print("raster :", RASTER.name)
print("aoi    :", AOI_FILE.name)
print("history:", HISTORY_CSV.name)

## 1. Resolve which bands hold RED and NIR

The demo scene carries band descriptions (B02/B03/B04/B08), so detection is exact.

In [ ]:
metadata = data.get_raster_metadata(RASTER)
resolution = indices.resolve_band_map(metadata, None)
print("band descriptions  :", metadata["band_descriptions"])
print("resolved bands     :", resolution["bands"])
print("assumption warning :", resolution["assumption_warning"] or "none")

manual = indices.resolve_band_map(metadata, {"red": 3, "nir": 4})
print("explicit override  :", manual["bands"])

## 2. Clip to the field and build a band stack

The clip is windowed (never a full read), the AOI becomes a pixel mask, and the stack keeps its transform so results stay georeferenced.

In [ ]:
aoi = data.load_aoi(AOI_FILE)

clipped = geospatial.clip_raster_to_aoi(
    str(RASTER),
    aoi["gdf"],
    bands=sorted(set(resolution["bands"].values())),
    max_pixels=config.max_analysis_pixels(),
)
stack = preprocessing.band_stack(clipped, resolution["bands"])
valid = np.asarray(stack["valid_mask"])

print("clip window     :", stack["shape"], "(source", clipped["source_shape"], ")")
print("decimated       :", stack["decimated"], "| factor", clipped["decimation"])
print("pixel area (ha) :", stack["pixel_area_ha"])
print("AOI coverage    :", round(float(np.asarray(clipped["mask"]).mean()) * 100, 2), "% of the window")
print("valid pixels    :", int(valid.sum()), "of", valid.size)
print("clip bounds     :", clipped["clip_bounds_lonlat"])

## 3. Compute NDVI

Division by zero, nodata and invalid pixels are handled inside the engine; the array is `float32` with `NaN` where no value exists.

In [ ]:
computed = indices.compute_index("NDVI", stack["bands"], valid=valid)
print("engine returned:", sorted(computed))

ndvi = np.asarray(computed["array"], dtype="float32")
print("range: %.4f .. %.4f | finite pixels: %d" % (np.nanmin(ndvi), np.nanmax(ndvi), int(np.isfinite(ndvi).sum())))

In [ ]:
statistics = indices.index_statistics(ndvi, valid=valid)
show = ("min", "max", "mean", "median", "std", "variance", "valid_pixels", "p5", "p25", "p75", "p95")
print(json.dumps({key: statistics[key] for key in show if key in statistics}, indent=2, default=str))

## 4. The vegetation map

The visualization engine renders the same map the dashboard shows, and writes it under `outputs/`.

In [ ]:
map_path = visualization.save_index_map(
    ndvi,
    NOTEBOOK_OUT / "03_ndvi_map.png",
    transform=stack["transform"],
    crs=stack["crs"],
    aoi_gdf=aoi["gdf"],
    bounds=clipped["clip_bounds"],
    valid=valid,
    index_name="NDVI",
    title="NDVI - synthetic demo field",
    caption="Computed in notebook 03 from the labelled synthetic scene.",
)
print("saved:", pathlib.Path(map_path).relative_to(ROOT))

masked = np.where(valid & np.isfinite(ndvi), ndvi, np.nan)
figure, axes = plt.subplots(figsize=(7, 6))
image = axes.imshow(masked, cmap="RdYlGn", vmin=-0.2, vmax=1.0)
figure.colorbar(image, ax=axes, label="NDVI", shrink=0.85)
axes.set_title("NDVI (red = bare soil / stress, green = dense canopy)")
plt.tight_layout()
plt.show()

## 5. Every index on the same stack

NDVI is the default, but NDRE, EVI and SAVI are one call away - the only difference is which bands they require.

In [ ]:
rows = []
for index_name in sorted(indices.INDEX_REGISTRY):
    required = indices.INDEX_REGISTRY[index_name]["bands"]
    if not all(band in stack["bands"] for band in required):
        continue
    values = np.asarray(indices.compute_index(index_name, stack["bands"], valid=valid)["array"], dtype="float32")
    stats = indices.index_statistics(values, valid=valid)
    rows.append({
        "index": index_name,
        "bands": ", ".join(required),
        "mean": round(stats["mean"], 4),
        "median": round(stats["median"], 4),
        "std": round(stats["std"], 4),
        "min": round(stats["min"], 4),
        "max": round(stats["max"], 4),
    })
pd.DataFrame(rows)

**What NDVI does and does not tell you.** A vegetation index measures canopy
vigour from reflectance. Low values point at sparse cover, bare soil, water or
stress - they never identify a specific disease or pest on their own. Use them
to prioritise scouting, then confirm in the field.